In [1]:
# If needed, run this once in your notebook
!pip install openai pandas python-dotenv tqdm

You should consider upgrading via the '/home/ec2-user/.venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import os
import time
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Mistral API (OpenAI-compatible)
api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    raise ValueError("MISTRAL_API_KEY not found in .env file")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.mistral.ai/v1",
)
print("Mistral API key loaded successfully.")

Mistral API key loaded successfully.


In [3]:
input_path = "../data/Ambivalent_RQ1_s1.2.csv"
df = pd.read_csv(input_path)

required_cols = ["title", "body", "parent_comment"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Fill missing values safely
for col in required_cols:
    df[col] = df[col].fillna("").astype(str)

print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(2)

Loaded shape: (550, 3)
Columns: ['title', 'body', 'parent_comment']


,title,body,parent_comment
0,Am I justified in thinking this was assualt?,"Alright so for context, I'm a dude but a trans...",yeah i’m not sure if the thigh incident is con...
1,I don't understand why I kept seeing my abuser...,"So, I was sexually assaulted and raped multipl...",similar sitch - one of my abusers was my child...


In [4]:
SYSTEM_PROMPT = """You are a counselor. You will read a sexual violence story (Title + Body) and a single comment responding to it.

Your task is to identify if your response to the story would be along the lines of the given comment. If you choose your response to be like the given comment, predict YES. If your response will be different, predict NO.

OUTPUT FORMAT (ONLY ONE WORD)
Yes
or
No"""

In [5]:
def classify_comment(title: str, body: str, parent_comment: str, max_retries: int = 3) -> str:
    user_input = f"""Title:
{title}

Body:
{body}

Comment:
{parent_comment}"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="mistral-large-latest",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_input},
                ],
                max_tokens=16,
                temperature=0,
            )

            raw_text = (response.choices[0].message.content or "").strip()
            txt = raw_text.lower().strip()

            # Strict normalization
            if txt == "yes":
                return "Yes"
            if txt == "no":
                return "No"

            # Fallback parsing if model adds extra words
            if "yes" in txt:
                return "Yes"
            if "no" in txt:
                return "No"

            return "PARSE_ERROR"

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                print(f"Error after retries: {e}")
                return "API_ERROR"

In [6]:
test_idx = 0

test_label = classify_comment(
    title=df.loc[test_idx, "title"],
    body=df.loc[test_idx, "body"],
    parent_comment=df.loc[test_idx, "parent_comment"]
)

print("Test label:", test_label)
print("\nTitle:", df.loc[test_idx, "title"][:200])
print("\nComment:", df.loc[test_idx, "parent_comment"][:300])

Test label: Yes

Title: Am I justified in thinking this was assualt?

Comment: yeah i’m not sure if the thigh incident is considered SA legally (it might be i’m not sure), but it sounds like if you hadn’t taken distance away from this person something bad could’ve happened, and it’s okay to feel gross about what happened. i am a cis woman, so it’s very different but imagine if


In [7]:
preds = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    label = classify_comment(
        title=row["title"],
        body=row["body"],
        parent_comment=row["parent_comment"]
    )
    preds.append(label)

df["usefulness_label"] = preds
print("Classification complete.")

Classifying:   0%|          | 0/550 [00:00<?, ?it/s]

Classifying: 100%|██████████| 550/550 [04:59<00:00,  1.84it/s]

Classification complete.


In [10]:
df["usefulness_label"].value_counts(dropna=False)

usefulness_label
Yes    307
No     243
Name: count, dtype: int64

In [9]:
output_path = "../data/Ambivalent_RQ1_yesno_mistral.csv"
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/Ambivalent_RQ1_yesno_mistral.csv


In [11]:
import pandas as pd

# Load predictions with Verdict column
analysis_df = pd.read_csv(output_path)

pred = analysis_df["usefulness_label"].astype(str).str.strip().str.lower()
verdict = analysis_df["Verdict"].astype(str).str.strip().str.lower()

yes_mask = pred == "yes"
no_mask = pred == "no"

yes_and_useful = (yes_mask & (verdict == "useful")).sum()
no_and_not_useful = (no_mask & (verdict == "not useful")).sum()

total_yes = yes_mask.sum()
total_no = no_mask.sum()

pct_yes_useful = (yes_and_useful / total_yes * 100) if total_yes > 0 else float("nan")
pct_no_not_useful = (no_and_not_useful / total_no * 100) if total_no > 0 else float("nan")

print(f"Yes & Useful: {yes_and_useful} / {total_yes}")
print(f"No & Not Useful: {no_and_not_useful} / {total_no}")
print(f"Yes and Useful / Total Yes: {pct_yes_useful:.2f}%")
print(f"No and Not Useful / Total No: {pct_no_not_useful:.2f}%")

Yes & Useful: 282 / 307
No & Not Useful: 46 / 243
Yes and Useful / Total Yes: 91.86%
No and Not Useful / Total No: 18.93%
